# POCUS-AI: Radiomics Feature Extraction

This notebook demonstrates how to use POCUS-AI's radiomics module to extract quantitative features from ultrasound images. Radiomics is the high-throughput extraction of quantitative features from medical images, providing a deeper analysis beyond what is visible to the human eye.

In [ ]:
# Import required libraries
import os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import seaborn as sns
from IPython.display import display, Markdown
import warnings
warnings.filterwarnings('ignore')  # Suppress warnings for cleaner output

# Add parent directory to path
import sys
sys.path.append('..')

# Try to import pyradiomics (not required for this demonstration)
try:
    import SimpleITK as sitk
    import pydicom
    import radiomics
    from radiomics import featureextractor
    RADIOMICS_AVAILABLE = True
except ImportError:
    RADIOMICS_AVAILABLE = False
    print("Note: PyRadiomics package not found. Some features may be limited.")
    print("Install with: pip install pyradiomics SimpleITK")

# Import POCUS-AI modules
from src.pocus_ai.radiomics import extract_features
from src.pocus_ai.utils.visualization import visualize_features

# Set random seed for reproducibility
np.random.seed(42)

## 1. Creating Sample Ultrasound Images

For this demonstration, we'll create synthetic ultrasound images with different patterns. In practice, you would use real DICOM files from ultrasound machines.

In [ ]:
def create_synthetic_ultrasound(size=(256, 256), pattern_type='normal'):
    """
    Create a synthetic ultrasound image with different patterns
    
    Args:
        size: Image dimensions
        pattern_type: Type of pattern to create ('normal', 'cyst', 'solid', 'complex')
        
    Returns:
        Synthetic ultrasound image as numpy array
    """
    # Create base image with gaussian noise (background)
    image = np.random.normal(0, 0.1, size)
    
    # Add ultrasound-like attenuation (gradual darkening with depth)
    y_gradient = np.linspace(1.0, 0.7, size[0])
    for y in range(size[0]):
        image[y, :] *= y_gradient[y]
    
    # Create different patterns
    x, y = np.ogrid[:size[0], :size[1]]
    center = (size[0]//2, size[1]//2)
    
    if pattern_type == 'normal':
        # Normal tissue pattern (heterogeneous)
        for i in range(20):
            cx = np.random.randint(size[0]//4, 3*size[0]//4)
            cy = np.random.randint(size[0]//4, 3*size[0]//4)
            r = np.random.randint(5, 20)
            mask = (x - cx)**2 + (y - cy)**2 <= r**2
            image[mask] += np.random.uniform(0.5, 1.5)
            
    elif pattern_type == 'cyst':
        # Cystic pattern (dark, well-defined circular area)
        r_outer = size[0]//5
        r_inner = size[0]//6
        outer_mask = (x - center[0])**2 + (y - center[1])**2 <= r_outer**2
        inner_mask = (x - center[0])**2 + (y - center[1])**2 <= r_inner**2
        
        # Add bright boundary
        image[outer_mask] += 2.0
        # Add dark center (cyst)
        image[inner_mask] = 0.1
        
    elif pattern_type == 'solid':
        # Solid mass (bright, heterogeneous area)
        r = size[0]//5
        mask = (x - center[0])**2 + (y - center[1])**2 <= r**2
        image[mask] += 2.0
        
        # Add internal heterogeneity
        for i in range(30):
            cx = center[0] + np.random.randint(-r//2, r//2)
            cy = center[1] + np.random.randint(-r//2, r//2)
            small_r = np.random.randint(2, 5)
            small_mask = (x - cx)**2 + (y - cy)**2 <= small_r**2
            image[small_mask] += np.random.uniform(-0.5, 0.5)
            
    elif pattern_type == 'complex':
        # Complex pattern (mixed solid and cystic components)
        r = size[0]//4
        mask = (x - center[0])**2 + (y - center[1])**2 <= r**2
        image[mask] += 1.5
        
        # Add cystic component
        cx = center[0] + size[0]//10
        cy = center[1] - size[0]//10
        cyst_r = size[0]//8
        cyst_mask = (x - cx)**2 + (y - cy)**2 <= cyst_r**2
        image[cyst_mask] = 0.1
        
    # Add speckle noise (characteristic of ultrasound)
    speckle = np.random.normal(0, 0.2, size)
    image += speckle
    
    # Normalize to [0, 1]
    image = (image - image.min()) / (image.max() - image.min())
    
    return image

# Generate sample images with different patterns
patterns = ['normal', 'cyst', 'solid', 'complex']
sample_images = {pattern: create_synthetic_ultrasound(pattern_type=pattern) for pattern in patterns}

# Display the images
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, (pattern, image) in enumerate(sample_images.items()):
    axes[i].imshow(image, cmap='gray')
    axes[i].set_title(f'Synthetic {pattern.capitalize()} Ultrasound')
    axes[i].axis('off')
    
plt.tight_layout()
plt.show()

## 2. Creating Region of Interest (ROI) Masks

In radiomics, we typically analyze a specific region of interest (ROI) within an image. Let's create masks for our sample images:

In [ ]:
def create_roi_mask(image, pattern_type='normal'):
    """Create a region of interest mask based on pattern type"""
    size = image.shape
    x, y = np.ogrid[:size[0], :size[1]]
    center = (size[0]//2, size[1]//2)
    mask = np.zeros(size, dtype=np.uint8)
    
    if pattern_type == 'normal':
        # ROI in central area
        r = size[0]//3
        roi = (x - center[0])**2 + (y - center[1])**2 <= r**2
        mask[roi] = 1
        
    elif pattern_type == 'cyst':
        # ROI around cystic area
        r_inner = size[0]//6
        roi = (x - center[0])**2 + (y - center[1])**2 <= r_inner**2
        mask[roi] = 1
        
    elif pattern_type == 'solid':
        # ROI covering solid mass
        r = size[0]//5
        roi = (x - center[0])**2 + (y - center[1])**2 <= r**2
        mask[roi] = 1
        
    elif pattern_type == 'complex':
        # ROI covering the entire complex area
        r = size[0]//4
        roi = (x - center[0])**2 + (y - center[1])**2 <= r**2
        mask[roi] = 1
    
    return mask

# Create ROI masks for each image
sample_masks = {pattern: create_roi_mask(image, pattern) 
                for pattern, image in sample_images.items()}

# Display images with masks overlaid
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, (pattern, image) in enumerate(sample_images.items()):
    # Create an RGB version of the image
    rgb_image = np.stack([image]*3, axis=2)
    
    # Create mask overlay (red)
    mask = sample_masks[pattern]
    rgb_image[:,:,0] = np.where(mask > 0, 1, rgb_image[:,:,0])
    rgb_image[:,:,1] = np.where(mask > 0, 0.5*rgb_image[:,:,1], rgb_image[:,:,1])
    rgb_image[:,:,2] = np.where(mask > 0, 0.5*rgb_image[:,:,2], rgb_image[:,:,2])
    
    axes[i].imshow(rgb_image)
    axes[i].set_title(f'{pattern.capitalize()} with ROI')
    axes[i].axis('off')
    
plt.tight_layout()
plt.show()

## 3. Extracting Radiomics Features

Now let's use POCUS-AI's radiomics module to extract features from our sample images:

In [ ]:
# Extract radiomics features from our sample images
feature_results = {}

for pattern, image in sample_images.items():
    mask = sample_masks[pattern]
    
    try:
        # Extract features using our module
        features = extract_features(image, mask)
        
        if 'error' in features:
            print(f"Error extracting features from {pattern} pattern: {features['error']}")
            continue
            
        feature_results[pattern] = features
        
        # Print summary of feature types
        feature_categories = {}
        for key in features.keys():
            if '_' in key:
                category = key.split('_')[0]
                if category not in feature_categories:
                    feature_categories[category] = 0
                feature_categories[category] += 1
                
        print(f"\n{pattern.capitalize()} pattern: Extracted {len(features)} features")
        for category, count in feature_categories.items():
            if category not in ['diagnostics', 'general']:
                print(f"  - {category}: {count} features")
                
    except Exception as e:
        if not RADIOMICS_AVAILABLE:
            print("\nNote: Full feature extraction requires PyRadiomics.")
            print("Using synthetic feature data for demonstration.")
            
            # Create synthetic features for demonstration
            feature_results[pattern] = {
                "firstorder_Mean": np.mean(image[mask > 0]),
                "firstorder_Std": np.std(image[mask > 0]),
                "firstorder_Entropy": -np.sum(image[mask > 0] * np.log2(image[mask > 0] + 1e-10)),
                "firstorder_Energy": np.sum(image[mask > 0]**2),
                "shape2D_Area": np.sum(mask),
                "shape2D_Perimeter": np.sum(mask),  # This is not actual perimeter, just for demo
                "glcm_Contrast": np.random.uniform(0.2, 0.8),
                "glcm_Correlation": np.random.uniform(0.3, 0.9),
                "glrlm_GrayLevelNonUniformity": np.random.uniform(100, 300)
            }
            
            print(f"\n{pattern.capitalize()} pattern: Created synthetic features for demonstration")
        else:
            print(f"Error extracting features from {pattern} pattern: {str(e)}")

## 4. Analyzing and Comparing Features

Now let's analyze and compare the radiomics features between different ultrasound patterns:

In [ ]:
# Select a subset of important features to analyze
important_features = [
    'firstorder_Mean', 
    'firstorder_Entropy', 
    'firstorder_Energy',
    'firstorder_Variance',
    'shape2D_Area',
    'glcm_Contrast',
    'glcm_Correlation'
]

# Create a dataframe for analysis
feature_df = []

for pattern, features in feature_results.items():
    row = {'Pattern': pattern}
    for feat in important_features:
        if feat in features:
            row[feat] = features[feat]
        else:
            # Handle missing features
            row[feat] = np.nan
    feature_df.append(row)
    
feature_df = pd.DataFrame(feature_df)
display(feature_df)

# Plot feature comparisons
plt.figure(figsize=(14, 8))

# Normalize features for better visualization
normalized_df = feature_df.copy()
for col in important_features:
    if col in normalized_df.columns:
        min_val = normalized_df[col].min()
        max_val = normalized_df[col].max()
        if max_val > min_val:
            normalized_df[col] = (normalized_df[col] - min_val) / (max_val - min_val)

# Create heatmap
if len(normalized_df) > 0:
    plt.subplot(1, 2, 1)
    heatmap_data = normalized_df.set_index('Pattern')
    sns.heatmap(heatmap_data, annot=True, cmap='viridis', fmt='.2f')
    plt.title('Normalized Feature Comparison')
    
    # Create bar plot for firstorder_Mean
    plt.subplot(1, 2, 2)
    if 'firstorder_Mean' in feature_df.columns:
        sns.barplot(x='Pattern', y='firstorder_Mean', data=feature_df)
        plt.title('Mean Intensity by Pattern')
        plt.ylabel('Mean Intensity')
        plt.xticks(rotation=45)
else:
    plt.text(0.5, 0.5, 'Not enough data for visualization', 
             ha='center', va='center', fontsize=14)

plt.tight_layout()
plt.show()

## 5. Feature Interpretation and Significance

Different radiomics features capture various aspects of the ultrasound image:

1. **First-order statistics**:
   - **Mean**: Average intensity within the ROI
   - **Variance**: How much pixel intensities vary
   - **Entropy**: Measure of intensity randomness/heterogeneity
   - **Energy**: Sum of squared intensities (uniform regions have higher energy)

2. **Shape features**:
   - **Area**: Size of the ROI
   - **Perimeter**: Boundary length
   - **Sphericity**: How closely the ROI resembles a sphere

3. **Texture features (GLCM)**:
   - **Contrast**: Local intensity variations
   - **Correlation**: Linear dependency of intensities
   - **Homogeneity**: Closeness of element distribution

4. **Clinical significance**:
   - Solid lesions typically show higher heterogeneity (entropy)
   - Cysts show lower mean intensity and lower entropy
   - Complex lesions often have mixed features

Let's summarize the characteristics of each pattern:

In [ ]:
# Provide textual interpretation based on features
pattern_interpretations = {
    'normal': 'Normal tissue typically shows moderate heterogeneity with speckle noise pattern characteristic of ultrasound. First-order statistics show moderate entropy and contrast.',
    
    'cyst': 'Cystic lesions typically appear as well-defined anechoic (dark) regions with enhanced posterior acoustic features. They show low mean intensity, low entropy (more homogeneous), and a well-defined shape.',
    
    'solid': 'Solid lesions appear as hyperechoic (bright) regions with internal heterogeneity. They show higher mean intensity, higher entropy, and potentially irregular shapes depending on the nature of the lesion.',
    
    'complex': 'Complex lesions contain both solid and cystic components, resulting in mixed echogenicity. Features show high heterogeneity (entropy) with a combination of features from both solid and cystic patterns.'
}

# Display interpretations
for pattern, interpretation in pattern_interpretations.items():
    display(Markdown(f"### {pattern.capitalize()}"))
    display(Markdown(interpretation))
    
    # Show some key features if available
    if pattern in feature_results:
        feature_text = "**Key features:**\n"
        for feat in ['firstorder_Mean', 'firstorder_Entropy', 'glcm_Contrast']:
            if feat in feature_results[pattern]:
                feature_name = feat.split('_')[1]
                value = feature_results[pattern][feat]
                feature_text += f"- {feature_name}: {value:.4f}\n"
        display(Markdown(feature_text))
    print()

## 6. Advanced Application: Machine Learning with Radiomics Features

Radiomics features can be used as input to machine learning models for tasks such as:
- Classification (e.g., benign vs. malignant lesions)
- Prediction of treatment response
- Disease progression monitoring

Let's demonstrate a simple classification model using our synthetic data:

In [ ]:
# Generate more samples for machine learning demonstration
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

# Generate a larger dataset (20 samples per pattern)
num_samples = 20
patterns = ['normal', 'cyst', 'solid', 'complex']

# Create a function to generate random feature variations
def generate_feature_variations(base_features, num_samples=20, variation=0.2):
    """Generate variations of base features for synthetic dataset"""
    samples = []
    for i in range(num_samples):
        sample = {}
        for key, value in base_features.items():
            if isinstance(value, (int, float)) and not key.startswith('general'):
                # Add random variation
                if key == 'firstorder_Mean':
                    # Keep variation within expected ranges for each pattern
                    if 'cyst' in sample.get('Pattern', ''):
                        sample[key] = value * np.random.uniform(0.8, 1.2)
                    elif 'solid' in sample.get('Pattern', ''):
                        sample[key] = value * np.random.uniform(0.9, 1.3)
                    else:
                        sample[key] = value * np.random.uniform(0.9, 1.1)
                else:
                    sample[key] = value * np.random.uniform(1-variation, 1+variation)
            else:
                sample[key] = value
        samples.append(sample)
    return samples

# Generate dataset
ml_data = []
for pattern in patterns:
    if pattern in feature_results:
        base_features = feature_results[pattern].copy()
        base_features['Pattern'] = pattern
        variations = generate_feature_variations(base_features, num_samples)
        ml_data.extend(variations)

# Convert to dataframe
ml_df = pd.DataFrame(ml_data)

# Select features for model
feature_cols = [col for col in ml_df.columns 
                if any(col.startswith(prefix) for prefix in 
                      ['firstorder_', 'glcm_', 'shape']) and 'diagnostics' not in col]

# Prepare data for modeling
X = ml_df[feature_cols].fillna(0)
y = ml_df['Pattern']

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train a random forest classifier
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)

# Make predictions
y_pred = rf.predict(X_test_scaled)

# Evaluate the model
print("Classification Report:")
print(classification_report(y_test, y_pred))

# Plot confusion matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=patterns, yticklabels=patterns)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

# Feature importance
feature_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_importance.head(10))
plt.title('Top 10 Most Important Features')
plt.tight_layout()
plt.show()

## 7. Practical Applications in Clinical Ultrasound

Radiomics analysis of ultrasound images has several practical clinical applications:

1. **Automated lesion classification**: Distinguishing between benign and malignant lesions
2. **Treatment response assessment**: Quantifying changes in lesion characteristics over time
3. **Computer-aided diagnosis**: Providing quantitative support for clinical decision making
4. **Standardized reporting**: Creating objective measures for ultrasound findings
5. **Research applications**: Discovering new imaging biomarkers for disease processes

The framework we've demonstrated can be extended to:
- Process real DICOM ultrasound images
- Extract features from specific anatomical regions
- Train models on larger datasets with known pathology
- Integrate with clinical decision support systems

## 8. Conclusion

Radiomics provides a powerful framework for quantitative analysis of ultrasound images, going beyond visual assessment to extract subtle features that may indicate pathology. The POCUS-AI package implements these capabilities with a focus on point-of-care ultrasound applications.

Next steps for implementation:
1. Apply to real ultrasound datasets with clinical annotations
2. Validate feature robustness across different ultrasound machines
3. Develop specialized models for specific clinical applications
4. Integrate with deep learning approaches for end-to-end analysis